# Bronze Core Source Ingestion

## tl;dr

This notebook wrote 13 source-oriented Parquet files for players, schedules, weekly player stats, weekly rosters, and snap counts across 2023–2025. The files contain 301,581 total source rows and occupy about 9 MB.

All files passed immediate shape-and-column round-trip checks with no duplicate candidate keys. Known null player keys were preserved: 22 weekly player-stat rows per season and 5, 7, and 18 weekly-roster rows for 2023, 2024, and 2025.

No rows or source columns are intentionally removed, and no Silver or Gold feature engineering is performed.

## Context & Methods

### Scope

- Seasons: 2023–2025
- Environment: `sports_dev_env`
- Source loader: `nflreadpy`
- In-memory format: pandas after explicit conversion from the loader's Polars output
- Storage format: Parquet
- Run location: repository root

Play-by-play is deferred to the next Bronze notebook because it is much larger. Participation remains optional for the initial WR slice.

## Setup

In [1]:
from pathlib import Path

import nflreadpy as nfl
import pandas as pd

# Keep the extraction scope explicit and easy to change for future refreshes.
SEASONS = [2023, 2024, 2025]

# Find the repository root whether this runs from Jupyter or nbconvert.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "ROADMAP.md").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "ROADMAP.md").exists():
            PROJECT_ROOT = parent
            break
    else:
        raise RuntimeError("Could not locate the repository root.")

BRONZE_DIR = PROJECT_ROOT / "data/bronze"

# Each ingestion section adds one record per file for the final summary.
output_records = []

## Data

## 1. Players

### What we are verifying

- The complete player source can be written without filtering columns or rows
- `gsis_id` remains unique and non-null
- The Parquet file reloads with the same shape and columns

In [2]:
players_dir = BRONZE_DIR / "players"
players_dir.mkdir(parents=True, exist_ok=True)
players_path = players_dir / "players.parquet"

# Player metadata is a single cross-season source rather than one file per season.
players = nfl.load_players().to_pandas()

# Measure key quality without altering or dropping any source rows.
player_key = ["gsis_id"]
player_key_null_rows = players[player_key].isna().any(axis=1).sum()
player_duplicate_rows = players.duplicated(player_key).sum()

# Persist the complete source frame, then verify the on-disk round trip.
players.to_parquet(players_path, index=False)
players_reloaded = pd.read_parquet(players_path)

assert players_reloaded.shape == players.shape
assert players_reloaded.columns.tolist() == players.columns.tolist()

# Save the validation results for the notebook-level output summary.
output_records.append(
    {
        "dataset": "players",
        "season": "all",
        "path": str(players_path),
        "rows": len(players),
        "columns": len(players.columns),
        "key": " + ".join(player_key),
        "null_key_rows": player_key_null_rows,
        "duplicate_key_rows": player_duplicate_rows,
        "file_mb": round(players_path.stat().st_size / 1024**2, 2),
        "round_trip_passed": True,
    }
)

pd.DataFrame([output_records[-1]])

,dataset,season,path,rows,columns,key,null_key_rows,duplicate_key_rows,file_mb,round_trip_passed
0,players,all,/Users/dtwice/Development/dev_sports/nfl/fanta...,24828,39,gsis_id,0,0,3.24,True


In [3]:
# Release source frames before loading the next dataset.
del players, players_reloaded

## 2. Schedules

### What we are verifying

- One source file is written for each requested season
- Each file contains only its requested season
- `game_id` is populated and unique
- Each Parquet file reloads with the same shape and columns

In [4]:
schedules_dir = BRONZE_DIR / "schedules"
schedules_dir.mkdir(parents=True, exist_ok=True)

# Remember where this section starts so its result table stays focused.
schedule_output_start = len(output_records)

# Process seasons independently so each file can be refreshed on its own.
for season in SEASONS:
    schedules_path = schedules_dir / f"schedules_{season}.parquet"
    schedules = nfl.load_schedules(seasons=[season]).to_pandas()

    # Confirm the loader honored the requested season before writing.
    assert set(schedules["season"].dropna().unique()) == {season}

    # Report source-key issues, but preserve the source exactly in Bronze.
    schedule_key = ["game_id"]
    schedule_key_null_rows = schedules[schedule_key].isna().any(axis=1).sum()
    schedule_duplicate_rows = schedules.duplicated(schedule_key).sum()

    # Write and immediately reload to catch serialization problems.
    schedules.to_parquet(schedules_path, index=False)
    schedules_reloaded = pd.read_parquet(schedules_path)

    assert schedules_reloaded.shape == schedules.shape
    assert schedules_reloaded.columns.tolist() == schedules.columns.tolist()

    output_records.append(
        {
            "dataset": "schedules",
            "season": season,
            "path": str(schedules_path),
            "rows": len(schedules),
            "columns": len(schedules.columns),
            "key": " + ".join(schedule_key),
            "null_key_rows": schedule_key_null_rows,
            "duplicate_key_rows": schedule_duplicate_rows,
            "file_mb": round(schedules_path.stat().st_size / 1024**2, 2),
            "round_trip_passed": True,
        }
    )

pd.DataFrame(output_records[schedule_output_start:])

,dataset,season,path,rows,columns,key,null_key_rows,duplicate_key_rows,file_mb,round_trip_passed
0,schedules,2023,/Users/dtwice/Development/dev_sports/nfl/fanta...,285,46,game_id,0,0,0.05,True
1,schedules,2024,/Users/dtwice/Development/dev_sports/nfl/fanta...,285,46,game_id,0,0,0.05,True
2,schedules,2025,/Users/dtwice/Development/dev_sports/nfl/fanta...,285,46,game_id,0,0,0.05,True


In [5]:
# Release the final loop frames before moving to the next source.
del schedules, schedules_reloaded

## 3. Weekly Player Stats

### What we are verifying

- Weekly player stats are requested explicitly with `summary_level="week"`
- One complete source file is written for each season
- The requested season is preserved
- The candidate player-week grain has no duplicate rows
- Null player keys remain in Bronze and are reported rather than removed
- Each Parquet file reloads with the same shape and columns

In [6]:
player_stats_dir = BRONZE_DIR / "player_stats_weekly"
player_stats_dir.mkdir(parents=True, exist_ok=True)

player_stats_output_start = len(output_records)

# Weekly stats are loaded and stored separately for each season.
for season in SEASONS:
    player_stats_path = (
        player_stats_dir / f"player_stats_weekly_{season}.parquet"
    )
    player_stats = nfl.load_player_stats(
        seasons=[season],
        summary_level="week",
    ).to_pandas()

    assert set(player_stats["season"].dropna().unique()) == {season}

    # Null player IDs are known source records; count them without removing them.
    player_stats_key = ["player_id", "season", "week", "season_type"]
    player_stats_key_null_rows = (
        player_stats[player_stats_key].isna().any(axis=1).sum()
    )
    player_stats_duplicate_rows = player_stats.duplicated(
        player_stats_key
    ).sum()

    # Validate that Parquet preserves the full weekly source shape and columns.
    player_stats.to_parquet(player_stats_path, index=False)
    player_stats_reloaded = pd.read_parquet(player_stats_path)

    assert player_stats_reloaded.shape == player_stats.shape
    assert (
        player_stats_reloaded.columns.tolist()
        == player_stats.columns.tolist()
    )

    output_records.append(
        {
            "dataset": "player_stats_weekly",
            "season": season,
            "path": str(player_stats_path),
            "rows": len(player_stats),
            "columns": len(player_stats.columns),
            "key": " + ".join(player_stats_key),
            "null_key_rows": player_stats_key_null_rows,
            "duplicate_key_rows": player_stats_duplicate_rows,
            "file_mb": round(player_stats_path.stat().st_size / 1024**2, 2),
            "round_trip_passed": True,
        }
    )

pd.DataFrame(output_records[player_stats_output_start:])

,dataset,season,path,rows,columns,key,null_key_rows,duplicate_key_rows,file_mb,round_trip_passed
0,player_stats_weekly,2023,/Users/dtwice/Development/dev_sports/nfl/fanta...,18643,150,player_id + season + week + season_type,22,0,0.84,True
1,player_stats_weekly,2024,/Users/dtwice/Development/dev_sports/nfl/fanta...,18983,150,player_id + season + week + season_type,22,0,0.84,True
2,player_stats_weekly,2025,/Users/dtwice/Development/dev_sports/nfl/fanta...,19422,150,player_id + season + week + season_type,22,0,0.84,True


In [7]:
# Release the final loop frames before moving to the next source.
del player_stats, player_stats_reloaded

## 4. Weekly Rosters

### What we are verifying

- One complete weekly-roster source file is written for each season
- The requested season is preserved
- The candidate player-team-week grain has no duplicate rows
- Null GSIS keys remain in Bronze and are reported rather than removed
- Each Parquet file reloads with the same shape and columns

In [8]:
rosters_dir = BRONZE_DIR / "rosters_weekly"
rosters_dir.mkdir(parents=True, exist_ok=True)

roster_output_start = len(output_records)

# Keep weekly roster snapshots season-specific for later player-team joins.
for season in SEASONS:
    rosters_path = rosters_dir / f"rosters_weekly_{season}.parquet"
    rosters = nfl.load_rosters_weekly(seasons=[season]).to_pandas()

    assert set(rosters["season"].dropna().unique()) == {season}

    # Preserve null GSIS records in Bronze while making their count visible.
    roster_key = ["gsis_id", "season", "week", "team"]
    roster_key_null_rows = rosters[roster_key].isna().any(axis=1).sum()
    roster_duplicate_rows = rosters.duplicated(roster_key).sum()

    # Validate the serialized file before treating it as a Bronze input.
    rosters.to_parquet(rosters_path, index=False)
    rosters_reloaded = pd.read_parquet(rosters_path)

    assert rosters_reloaded.shape == rosters.shape
    assert rosters_reloaded.columns.tolist() == rosters.columns.tolist()

    output_records.append(
        {
            "dataset": "rosters_weekly",
            "season": season,
            "path": str(rosters_path),
            "rows": len(rosters),
            "columns": len(rosters.columns),
            "key": " + ".join(roster_key),
            "null_key_rows": roster_key_null_rows,
            "duplicate_key_rows": roster_duplicate_rows,
            "file_mb": round(rosters_path.stat().st_size / 1024**2, 2),
            "round_trip_passed": True,
        }
    )

pd.DataFrame(output_records[roster_output_start:])

,dataset,season,path,rows,columns,key,null_key_rows,duplicate_key_rows,file_mb,round_trip_passed
0,rosters_weekly,2023,/Users/dtwice/Development/dev_sports/nfl/fanta...,45655,36,gsis_id + season + week + team,5,0,0.77,True
1,rosters_weekly,2024,/Users/dtwice/Development/dev_sports/nfl/fanta...,46579,36,gsis_id + season + week + team,7,0,0.80,True
2,rosters_weekly,2025,/Users/dtwice/Development/dev_sports/nfl/fanta...,46849,36,gsis_id + season + week + team,18,0,0.82,True


In [9]:
# Release the final loop frames before moving to the next source.
del rosters, rosters_reloaded

## 5. Snap Counts

### What we are verifying

- One complete snap-count source file is written for each season
- The requested season is preserved
- The candidate player-game-team grain is populated and unique
- Each Parquet file reloads with the same shape and columns

In [10]:
snap_counts_dir = BRONZE_DIR / "snap_counts"
snap_counts_dir.mkdir(parents=True, exist_ok=True)

snap_output_start = len(output_records)

# Store snap counts by season; Silver will later bridge PFR IDs to GSIS IDs.
for season in SEASONS:
    snap_counts_path = snap_counts_dir / f"snap_counts_{season}.parquet"
    snap_counts = nfl.load_snap_counts(seasons=[season]).to_pandas()

    assert set(snap_counts["season"].dropna().unique()) == {season}

    # Check the observed source grain without resolving identifiers yet.
    snap_counts_key = ["pfr_player_id", "game_id", "team"]
    snap_counts_key_null_rows = (
        snap_counts[snap_counts_key].isna().any(axis=1).sum()
    )
    snap_counts_duplicate_rows = snap_counts.duplicated(
        snap_counts_key
    ).sum()

    # Validate the serialized file before treating it as a Bronze input.
    snap_counts.to_parquet(snap_counts_path, index=False)
    snap_counts_reloaded = pd.read_parquet(snap_counts_path)

    assert snap_counts_reloaded.shape == snap_counts.shape
    assert (
        snap_counts_reloaded.columns.tolist()
        == snap_counts.columns.tolist()
    )

    output_records.append(
        {
            "dataset": "snap_counts",
            "season": season,
            "path": str(snap_counts_path),
            "rows": len(snap_counts),
            "columns": len(snap_counts.columns),
            "key": " + ".join(snap_counts_key),
            "null_key_rows": snap_counts_key_null_rows,
            "duplicate_key_rows": snap_counts_duplicate_rows,
            "file_mb": round(snap_counts_path.stat().st_size / 1024**2, 2),
            "round_trip_passed": True,
        }
    )

pd.DataFrame(output_records[snap_output_start:])

,dataset,season,path,rows,columns,key,null_key_rows,duplicate_key_rows,file_mb,round_trip_passed
0,snap_counts,2023,/Users/dtwice/Development/dev_sports/nfl/fanta...,26540,16,pfr_player_id + game_id + team,0,0,0.23,True
1,snap_counts,2024,/Users/dtwice/Development/dev_sports/nfl/fanta...,26615,16,pfr_player_id + game_id + team,0,0,0.23,True
2,snap_counts,2025,/Users/dtwice/Development/dev_sports/nfl/fanta...,26612,16,pfr_player_id + game_id + team,0,0,0.23,True


In [11]:
# Release the final loop frames; only the compact summary remains in memory.
del snap_counts, snap_counts_reloaded

## Results

## 6. Bronze Output Summary

This summary confirms what was written and whether each file passed its immediate Parquet round-trip check.

In [12]:
# Combine the per-file checks into one final ingestion report.
bronze_output_summary = (
    pd.DataFrame(output_records)
    .sort_values(["dataset", "season"])
    .reset_index(drop=True)
)

display(bronze_output_summary)

print(f"Files written: {len(bronze_output_summary)}")
print(f"Total rows written: {bronze_output_summary['rows'].sum():,}")
print(f"Total Parquet size: {bronze_output_summary['file_mb'].sum():,.2f} MB")
print(
    "All round trips passed:",
    bronze_output_summary["round_trip_passed"].all(),
)

,dataset,season,path,rows,columns,key,null_key_rows,duplicate_key_rows,file_mb,round_trip_passed
0,player_stats_weekly,2023,/Users/dtwice/Development/dev_sports/nfl/fanta...,18643,150,player_id + season + week + season_type,22,0,0.84,True
1,player_stats_weekly,2024,/Users/dtwice/Development/dev_sports/nfl/fanta...,18983,150,player_id + season + week + season_type,22,0,0.84,True
2,player_stats_weekly,2025,/Users/dtwice/Development/dev_sports/nfl/fanta...,19422,150,player_id + season + week + season_type,22,0,0.84,True
3,players,all,/Users/dtwice/Development/dev_sports/nfl/fanta...,24828,39,gsis_id,0,0,3.24,True
4,rosters_weekly,2023,/Users/dtwice/Development/dev_sports/nfl/fanta...,45655,36,gsis_id + season + week + team,5,0,0.77,True
5,rosters_weekly,2024,/Users/dtwice/Development/dev_sports/nfl/fanta...,46579,36,gsis_id + season + week + team,7,0,0.80,True
6,rosters_weekly,2025,/Users/dtwice/Development/dev_sports/nfl/fanta...,46849,36,gsis_id + season + week + team,18,0,0.82,True
7,schedules,2023,/Users/dtwice/Development/dev_sports/nfl/fanta...,285,46,game_id,0,0,0.05,True
8,schedules,2024,/Users/dtwice/Development/dev_sports/nfl/fanta...,285,46,game_id,0,0,0.05,True
9,schedules,2025,/Users/dtwice/Development/dev_sports/nfl/fanta...,285,46,game_id,0,0,0.05,True


Files written: 13
Total rows written: 301,581
Total Parquet size: 8.99 MB
All round trips passed: True


## Takeaways

- The core non-PBP Bronze sources are stored as complete source-oriented Parquet files.
- Data is split by season where the source accepts a season parameter.
- Known null player identifiers are preserved rather than silently removed.
- No source joins, player identity resolution, column renaming, or fantasy feature engineering occurs in this notebook.
- The next Bronze notebook should ingest play-by-play one season at a time.